In [ ]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader scikit-learn xgboost lightgbm catboost xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 8.5 MB/s eta 0:00:00


In [ ]:
import sys
!{sys.executable} -m pip install -q imblearn

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
import os
import re
import time
import json
import pickle
import warnings
from pathlib import Path
from typing import Dict, List
from collections import defaultdict
from itertools import combinations

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    r2_score,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE


# =============================================================================
# CONFIG
# =============================================================================
#
# OBJECTIF DE CE NOTEBOOK :
#   1. Pour chaque régime (CALM/NORMAL/STRESS/GLOBAL), tester 11 fenêtres de
#      train (TRAIN_START = 2000..2010) et calculer un score de qualité des
#      features sélectionnées (moyenne des |corrélations Spearman| des
#      features retenues par select_and_filter_features).
#   2. Pour chaque régime, garder la fenêtre TRAIN_START qui maximise ce
#      score (potentiellement différente d'un régime à l'autre).
#   3. Entraîner les 5 algos (XGBoost, LightGBM, GradientBoosting,
#      RandomForest, LogisticRegression) x N=1..20 features, UNIQUEMENT sur
#      la fenêtre gagnante de chaque régime, avec SMOTE appliqué partout.
#   4. Sortie : un seul fichier Excel récapitulatif.

OUTPUT_DIR = Path("/content/outputs_v18_best_window_per_regime")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Fenêtre de chargement la plus large possible : on télécharge une seule fois
# depuis la plus ancienne date candidate, puis chaque test de fenêtre filtre dessus.
MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START = "2000-01-01"   # utilisé uniquement pour le téléchargement initial
TEST_DATE = "2024-01-01"     # fixe pour toutes les phases

# --- Phase 1 : fenêtres candidates pour la sélection de la "meilleure fenêtre"
# par régime (sur le score de qualité des features, pas sur l'AUC test).
TRAIN_START_CANDIDATES = [f"{year}-01-01" for year in range(2000, 2011)]  # 2000..2010

FEATURE_PREFILTER_TOP_N = 300

YF_CHUNK_SIZE = 40
SLEEP_BETWEEN_CHUNKS = 1.0

MIN_COLUMN_COVERAGE = 0.90  # seuil de couverture par colonne, sur toute la fenêtre depuis MIN_DATA_START
MAX_FIRST_VALID_LAG_DAYS = 365

VIX_FLAT_PCT_THRESHOLD = 0.0

# Quantile de régime : mode fixed uniquement pour ce notebook (simplicité,
# cohérent avec le fait qu'on compare déjà 11 fenêtres de train différentes)
ROLLING_QUANTILE_WINDOW = 504

np.random.seed(RANDOM_STATE)

FRED_API_KEY = os.getenv("FRED_API_KEY")
if FRED_API_KEY:
    os.environ["FRED_API_KEY"] = FRED_API_KEY


In [ ]:
# =============================================================================
# Helper for Feature Selection
# =============================================================================

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    print("[FEATURE SELECTION] Starting feature selection...")

    # Calcul des corrélations avec la cible
    # Ensure all_features are in df columns and drop rows with NaNs for correlation calculation
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs. Cannot perform feature selection.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    # Filtrer les features avec une corrélation minimale avec la cible
    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    print(f"[FEATURE SELECTION] Features with correlation >= {correlation_threshold_target} with target: {len(selected_features_initial)}")

    if not selected_features_initial:
        warnings.warn("No features found with sufficient correlation to the target.")
        return []

    # Réduction de la redondance des features fortement corrélées entre elles
    # On ne garde que celles qui ont la plus forte corrélation à la target
    features_small = []
    high_corr_features = list(selected_features_initial)

    # Need to make sure df_for_corr still contains these features
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy() # Ensure these columns exist before calculating inter-correlation

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        # Prendre la feature la plus corrélée à la cible parmi les restantes
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        # Supprimer toutes les features fortement corrélées à 'f'
        correlated_with_f = []
        if f in df_for_inter_corr.columns: # Check if 'f' exists in df_for_inter_corr
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns: # Check if 'f_other' exists
                    # Calculate correlation only for existing columns
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception as e:
                        warnings.warn(f"Could not calculate correlation between {f} and {f_other}: {e}")

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    print(f"[FEATURE SELECTION] Final features after filtering inter-correlation: {len(features_small)}")

    return features_small

In [ ]:
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [ ]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [ ]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [ ]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [ ]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [ ]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        future_vix = vix.shift(-1)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df


In [ ]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [ ]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [ ]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [ ]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [ ]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)
print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)

[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40
[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LVRK"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CBOT_W"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['CBOT_W']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 7/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BZF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2026-06-29)')


[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [ ]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [ ]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6732, 1180), features: 985


In [ ]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [ ]:
# =============================================================================
# 5 algos (LogisticRegression réintégrée) testés pour chaque régime.
# SMOTE appliqué SYSTÉMATIQUEMENT et sans condition sur les 4 régimes
# (CALM/NORMAL/STRESS/GLOBAL) - voir la boucle d'entraînement en Phase 2.
# =============================================================================

ALL_REGIMES = ["CALM", "NORMAL", "STRESS", "GLOBAL"]
ALL_ALGOS = ["XGBoost", "LightGBM", "GradientBoosting", "RandomForest", "LogisticRegression"]

MAX_FEATURES_TO_TEST = 20  # phase 2 : on teste N=1..20 sur la fenêtre gagnante


In [ ]:
# =============================================================================
# PHASE 1 : pour chaque (train_start, régime), calculer le score de qualité
# des features = moyenne des |corrélations Spearman| des features retenues
# par select_and_filter_features sur le train de ce régime, pour cette fenêtre.
# Pas d'entraînement de modèle ici, juste la sélection de features + scoring.
# =============================================================================

def get_regime_train_slice(df_window: pd.DataFrame, regime: str, mode: str = "fixed"):
    """Construit le régime sur df_window et retourne le sous-dataframe TRAIN
    correspondant au régime demandé (ou tout le train si regime == GLOBAL)."""
    target_builder = TargetBuilder(q_low=0.33, q_high=0.67, mode=mode,
                                    rolling_window=ROLLING_QUANTILE_WINDOW)
    df = target_builder.build(df_window, train_end=TEST_DATE)

    train_mask = df.index < pd.Timestamp(TEST_DATE)
    df_train = df.loc[train_mask].copy()

    if regime == "GLOBAL":
        return df_train, df
    else:
        return df_train.loc[df_train["VIX_Regime"] == regime].copy(), df


print("="*80)
print("PHASE 1 : score de qualité des features par (Train_Start, Régime)")
print("="*80)

feature_quality_rows = []
# Cache : pour chaque (train_start) on garde le ranked_features par régime,
# pour éviter de refaire deux fois select_and_filter_features au moment de la
# phase 2 sur la fenêtre gagnante.
ranked_features_cache = {}  # clé = (train_start, regime) -> liste ordonnée de features

for train_start in TRAIN_START_CANDIDATES:
    df_window = df_post_features.loc[df_post_features.index >= pd.Timestamp(train_start)].copy()

    for regime in ALL_REGIMES:
        df_train_regime, _ = get_regime_train_slice(df_window, regime, mode="fixed")

        feats_all = [f for f in features_post_engineering if f in df_train_regime.columns]

        if len(df_train_regime) < 30 or "VIX_Direction" not in df_train_regime.columns:
            print(f"[WARN] train_start={train_start} régime={regime}: échantillon trop petit. Skip.")
            continue

        ranked_features = select_and_filter_features(
            df=df_train_regime,
            all_features=feats_all,
            target_col="VIX_Direction",
            correlation_threshold_target=correlation_threshold_target,
            correlation_threshold_features=correlation_threshold_features,
            max_features_to_select=MAX_FEATURES_TO_TEST,
        )

        if not ranked_features:
            print(f"[WARN] train_start={train_start} régime={regime}: aucune feature sélectionnée. Skip.")
            continue

        # Score = moyenne des |corrélation Spearman| des features retenues
        df_corr = df_train_regime[ranked_features + ["VIX_Direction"]].dropna()
        correlations = df_corr.corr(method="spearman")["VIX_Direction"].abs()
        correlations = correlations.drop("VIX_Direction", errors="ignore")
        quality_score = correlations.mean()

        ranked_features_cache[(train_start, regime)] = ranked_features

        feature_quality_rows.append({
            "Train_Start": train_start,
            "VIX_Regime": regime,
            "N_Features_Selected": len(ranked_features),
            "Quality_Score_MeanAbsCorr": quality_score,
            "Train_N": len(df_train_regime),
        })

        print(f"[SCORE] train_start={train_start} régime={regime}: "
              f"{len(ranked_features)} features, score={quality_score:.4f}, n={len(df_train_regime)}")

feature_quality_df = pd.DataFrame(feature_quality_rows)
feature_quality_df.to_csv(OUTPUT_DIR / "phase1_feature_quality_by_window.csv", index=False)
print(f"\n[SAVE] phase1_feature_quality_by_window.csv ({len(feature_quality_df)} lignes)")


PHASE 1 : score de qualité des features par (Train_Start, Régime)
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.04, NORMAL [15.04-21.34), STRESS >= 21.34  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6731
[TARGET] Remaining non-flat rows: 6475
VIX_Regime
NORMAL    2347
CALM      2089
STRESS    2039
Name: count, dtype: int64
[SCORE] train_start=2000-01-01 régime=CALM: 20 features, score=0.0928, n=1919
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.04, NORMAL [15.04-21.34), STRESS >= 21.34  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 256/6731
[TARGET] Remaining non-flat rows: 6475
VIX_Regime
NORMAL    2347
CALM      2089
STRESS    2039
Name: count, dtype: int64
[SCORE] train_start=2000-01-01 régime=NORMAL: 20 features, score=0.0889, n=1988
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGE

In [ ]:
# =============================================================================
# Détermination du Train_Start GAGNANT pour chaque régime (indépendamment) :
# celui qui maximise Quality_Score_MeanAbsCorr.
# =============================================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("[PHASE 1 RESULTS] Score par fenêtre et régime :")
pivot_quality = feature_quality_df.pivot_table(
    index="Train_Start", columns="VIX_Regime", values="Quality_Score_MeanAbsCorr"
)
display(pivot_quality)

winning_window_by_regime = {}
for regime in ALL_REGIMES:
    sub = feature_quality_df[feature_quality_df["VIX_Regime"] == regime]
    if sub.empty:
        print(f"[WARN] Aucune donnée pour {regime}, impossible de désigner un gagnant.")
        continue
    best_row = sub.loc[sub["Quality_Score_MeanAbsCorr"].idxmax()]
    winning_window_by_regime[regime] = best_row["Train_Start"]
    print(f"[WINNER] {regime}: Train_Start={best_row['Train_Start']} "
          f"(score={best_row['Quality_Score_MeanAbsCorr']:.4f}, "
          f"{int(best_row['N_Features_Selected'])} features, n={int(best_row['Train_N'])})")

winner_summary_df = pd.DataFrame([
    {"VIX_Regime": regime, "Winning_Train_Start": ts}
    for regime, ts in winning_window_by_regime.items()
])
winner_summary_df.to_csv(OUTPUT_DIR / "phase1_winning_window_per_regime.csv", index=False)
print(f"\n[SAVE] phase1_winning_window_per_regime.csv")
display(winner_summary_df)


[PHASE 1 RESULTS] Score par fenêtre et régime :


VIX_Regime,CALM,GLOBAL,NORMAL,STRESS
Train_Start,,,,
2000-01-01,0.092850,0.088213,0.088855,0.073152
2001-01-01,0.088086,0.088213,0.091129,0.075323
2002-01-01,0.089366,0.089741,0.094589,0.080071
2003-01-01,0.090774,0.095259,0.086696,0.086774
2004-01-01,0.091380,0.094690,0.082930,0.087620
2005-01-01,0.093086,0.094146,0.082439,0.084855
2006-01-01,0.102940,0.097613,0.086576,0.082169
2007-01-01,0.108499,0.099201,0.094096,0.081365
2008-01-01,0.107283,0.099018,0.096420,0.084183


[WINNER] CALM: Train_Start=2007-01-01 (score=0.1085, 20 features, n=1399)
[WINNER] NORMAL: Train_Start=2008-01-01 (score=0.0964, 20 features, n=1360)
[WINNER] STRESS: Train_Start=2010-01-01 (score=0.1023, 20 features, n=1161)
[WINNER] GLOBAL: Train_Start=2010-01-01 (score=0.1086, 20 features, n=3506)

[SAVE] phase1_winning_window_per_regime.csv


,VIX_Regime,Winning_Train_Start
0,CALM,2007-01-01
1,NORMAL,2008-01-01
2,STRESS,2010-01-01
3,GLOBAL,2010-01-01


In [ ]:
# =============================================================================
# PHASE 2 : entraînement final.
# Pour CHAQUE régime, sur SA fenêtre gagnante (winning_window_by_regime),
# on entraîne les 5 algos x N=1..20 features (liste déjà calculée en Phase 1,
# réutilisée via ranked_features_cache - pas de re-calcul), avec SMOTE
# appliqué systématiquement.
# =============================================================================

print("="*80)
print("PHASE 2 : entraînement final sur la fenêtre gagnante de chaque régime")
print("="*80)

# On doit reconstruire X_train_scaled / X_test_scaled / df_train / df_test
# SPÉCIFIQUEMENT pour la fenêtre gagnante de CHAQUE régime, puisque les
# fenêtres peuvent différer d'un régime à l'autre (donc des cleaners/scalers
# différents par régime, fittés chacun sur sa propre fenêtre).

final_rows = []
model_number = 0

for regime in ALL_REGIMES:
    train_start = winning_window_by_regime.get(regime)
    if train_start is None:
        print(f"[WARN] {regime}: pas de fenêtre gagnante désignée. Skip.")
        continue

    print(f"\n--- {regime}: fenêtre gagnante Train_Start={train_start} ---")

    df_window = df_post_features.loc[df_post_features.index >= pd.Timestamp(train_start)].copy()

    target_builder = TargetBuilder(q_low=0.33, q_high=0.67, mode="fixed",
                                    rolling_window=ROLLING_QUANTILE_WINDOW)
    df = target_builder.build(df_window, train_end=TEST_DATE)
    feats_all = [f for f in features_post_engineering if f in df.columns]
    df[feats_all] = df[feats_all].replace([np.inf, -np.inf], np.nan)

    train_mask = df.index < pd.Timestamp(TEST_DATE)
    test_mask = ~train_mask
    df_train = df.loc[train_mask].copy()
    df_test = df.loc[test_mask].copy()

    if len(df_train) == 0 or len(df_test) == 0:
        print(f"[WARN] {regime}: train ou test vide sur cette fenêtre. Skip.")
        continue

    cleaner = TrainFittedCleaner()
    X_train_clean = cleaner.fit_transform(df_train[feats_all])
    X_test_clean = cleaner.transform(df_test[feats_all])

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_clean), columns=feats_all, index=df_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_clean), columns=feats_all, index=df_test.index)
    y_train_series = pd.Series(df_train["VIX_Direction"].values, index=df_train.index)
    y_test_series = pd.Series(df_test["VIX_Direction"].values, index=df_test.index)

    if regime == "GLOBAL":
        train_mask_regime = pd.Series(True, index=df_train.index)
        test_mask_regime = pd.Series(True, index=df_test.index)
    else:
        train_mask_regime = df_train["VIX_Regime"] == regime
        test_mask_regime = df_test["VIX_Regime"] == regime

    y_train_regime_raw_full = y_train_series.loc[train_mask_regime].values
    X_test_regime_full = X_test_scaled.loc[test_mask_regime]
    y_test_regime = y_test_series.loc[test_mask_regime].values

    if len(y_train_regime_raw_full) < 30 or len(y_test_regime) < 10 or len(np.unique(y_train_regime_raw_full)) < 2:
        print(f"[WARN] {regime}: échantillon trop petit sur la fenêtre gagnante. Skip.")
        continue

    # Réutilise la liste de features déjà calculée en Phase 1 pour ce
    # (train_start, regime) exact - pas de nouveau select_and_filter_features.
    ranked_features = ranked_features_cache.get((train_start, regime))
    if not ranked_features:
        print(f"[WARN] {regime}: pas de features en cache pour {train_start}. Skip.")
        continue

    max_n = min(MAX_FEATURES_TO_TEST, len(ranked_features))
    configs = model_configs()

    for algo_name in ALL_ALGOS:
        Model_class, param_grid, fixed_args = configs[algo_name]

        for n in range(1, max_n + 1):
            feats_n = ranked_features[:n]

            X_train_regime = X_train_scaled.loc[train_mask_regime, feats_n].copy()
            y_train_regime_raw = y_train_regime_raw_full.copy()
            X_test_regime = X_test_regime_full[feats_n].copy()

            # SMOTE systématique (les 4 régimes)
            sm = SMOTE(random_state=RANDOM_STATE)
            unique_classes, counts = np.unique(y_train_regime_raw, return_counts=True)
            if len(unique_classes) > 1 and min(counts) > 1:
                X_train_resampled, y_train_regime = sm.fit_resample(X_train_regime, y_train_regime_raw)
                X_train_regime = pd.DataFrame(X_train_resampled, columns=feats_n)
            else:
                y_train_regime = y_train_regime_raw

            cv = TimeSeriesSplit(n_splits=3)
            try:
                grid = GridSearchCV(Model_class(**fixed_args), param_grid=param_grid,
                                     scoring="roc_auc", cv=cv, n_jobs=-1)
                grid.fit(X_train_regime.values, y_train_regime)
                model = grid.best_estimator_
            except Exception as e:
                print(f"[WARN] {regime}/{algo_name}/N={n}: échec GridSearchCV ({e}). Skip.")
                continue

            pred = model.predict(X_test_regime.values)
            proba = (model.predict_proba(X_test_regime.values)[:, 1]
                     if hasattr(model, "predict_proba") else pred.astype(float))
            metrics = compute_metrics(y_test_regime, pred, proba)

            model_number += 1
            row = {
                "Model_Number": model_number,
                "Model": algo_name,
                "VIX_Regime": regime,
                "Train_Start_Used": train_start,
                "Features": json.dumps(feats_n),
                "N_Features": n,
                "SMOTE_Used": True,
            }
            row.update(metrics)
            final_rows.append(row)

        print(f"  [{regime}/{algo_name}] N=1..{max_n} testés.")

final_results_df = pd.DataFrame(final_rows)
print(f"\n[PHASE 2 DONE] {len(final_results_df)} lignes générées au total.")


PHASE 2 : entraînement final sur la fenêtre gagnante de chaque régime

--- CALM: fenêtre gagnante Train_Start=2007-01-01 ---
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.11, NORMAL [15.11-21.23), STRESS >= 21.23  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 186/5069
[TARGET] Remaining non-flat rows: 4883
VIX_Regime
NORMAL    1800
CALM      1574
STRESS    1509
Name: count, dtype: int64
  [CALM/XGBoost] N=1..20 testés.
  [CALM/LightGBM] N=1..20 testés.
  [CALM/GradientBoosting] N=1..20 testés.
  [CALM/RandomForest] N=1..20 testés.
  [CALM/LogisticRegression] N=1..20 testés.

--- NORMAL: fenêtre gagnante Train_Start=2008-01-01 ---
[TARGET] VIX column: VIX_Price  |  mode=fixed
[TARGET] VIX regime thresholds: CALM < 15.22, NORMAL [15.22-21.31), STRESS >= 21.31  (fixe, calculé sur train)
[TARGET] VIX flat threshold: ±0.00%
[TARGET] Flat days removed completely: 178/4809
[TARGET] Remaining non-f

In [ ]:
# =============================================================================
# Résumé : meilleur modèle (par AUC) pour chaque régime, sur sa fenêtre
# gagnante.
# =============================================================================

best_per_regime_rows = []
for regime in ALL_REGIMES:
    sub = final_results_df[final_results_df["VIX_Regime"] == regime]
    if sub.empty:
        continue
    best_row = sub.loc[sub["AUC"].idxmax()]
    best_per_regime_rows.append(best_row)
    print(f"[BEST] {regime}: {best_row['Model']} N={best_row['N_Features']} "
          f"(Train_Start={best_row['Train_Start_Used']}) -> "
          f"AUC={best_row['AUC']:.4f} Acc={best_row['Accuracy']:.4f} F1={best_row['F1']:.4f}")

best_per_regime_df = pd.DataFrame(best_per_regime_rows)
display(best_per_regime_df[["VIX_Regime","Model","Train_Start_Used","N_Features",
                              "Accuracy","Precision","Recall","F1","AUC"]])


[BEST] CALM: RandomForest N=19 (Train_Start=2007-01-01) -> AUC=0.6315 Acc=0.6000 F1=0.5833
[BEST] NORMAL: LogisticRegression N=1 (Train_Start=2008-01-01) -> AUC=0.5959 Acc=0.4870 F1=0.5917
[BEST] STRESS: LightGBM N=18 (Train_Start=2010-01-01) -> AUC=0.5817 Acc=0.5674 F1=0.4299
[BEST] GLOBAL: GradientBoosting N=6 (Train_Start=2010-01-01) -> AUC=0.6099 Acc=0.5806 F1=0.5113


,VIX_Regime,Model,Train_Start_Used,N_Features,Accuracy,Precision,Recall,F1,AUC
78,CALM,RandomForest,2007-01-01,19,0.600000,0.653333,0.526882,0.583333,0.631524
180,NORMAL,LogisticRegression,2008-01-01,1,0.487032,0.451049,0.860000,0.591743,0.595905
237,STRESS,LightGBM,2010-01-01,18,0.567376,0.383333,0.489362,0.429907,0.581711
345,GLOBAL,GradientBoosting,2010-01-01,6,0.580645,0.531250,0.492754,0.511278,0.609928


In [ ]:
# =============================================================================
# UN SEUL fichier Excel final, avec 4 feuilles :
#   - Phase1_Feature_Scores : score de qualité des features par (Train_Start, régime)
#   - Phase1_Winning_Windows : la fenêtre gagnante retenue par régime
#   - Phase2_All_Results : tous les modèles entraînés (5 algos x N=1..20 x 4 régimes)
#   - Best_Model_Per_Regime : le meilleur modèle (AUC) par régime
# =============================================================================

excel_filename = OUTPUT_DIR / "vix_best_window_per_regime_report.xlsx"

with pd.ExcelWriter(excel_filename, engine="xlsxwriter") as writer:
    feature_quality_df.to_excel(writer, sheet_name="Phase1_Feature_Scores", index=False)
    winner_summary_df.to_excel(writer, sheet_name="Phase1_Winning_Windows", index=False)
    final_results_df.to_excel(writer, sheet_name="Phase2_All_Results", index=False)
    best_per_regime_df.to_excel(writer, sheet_name="Best_Model_Per_Regime", index=False)

print(f"[SAVE] Fichier Excel unique -> {excel_filename}")
print(f"[RÉCAP]")
print(f"  - Phase1_Feature_Scores  : {len(feature_quality_df)} lignes (11 fenêtres x 4 régimes)")
print(f"  - Phase1_Winning_Windows : {len(winner_summary_df)} lignes (1 par régime)")
print(f"  - Phase2_All_Results     : {len(final_results_df)} lignes (5 algos x N x régimes, sur fenêtre gagnante)")
print(f"  - Best_Model_Per_Regime  : {len(best_per_regime_df)} lignes (1 par régime)")


[SAVE] Fichier Excel unique -> /content/outputs_v18_best_window_per_regime/vix_best_window_per_regime_report.xlsx
[RÉCAP]
  - Phase1_Feature_Scores  : 44 lignes (11 fenêtres x 4 régimes)
  - Phase1_Winning_Windows : 4 lignes (1 par régime)
  - Phase2_All_Results     : 400 lignes (5 algos x N x régimes, sur fenêtre gagnante)
  - Best_Model_Per_Regime  : 4 lignes (1 par régime)
